# Imports

In [19]:
import pandas as pd
import numpy as np
import igraph as ig
from collections import defaultdict
import os
import itertools


# Part 1

In [147]:
df = pd.read_csv("data/Part_A/1/balanced_graph.csv")
g = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
g.es["sign"] = df["sign"].tolist()
triangles = g.cliques(min=3, max=3)

In [148]:
zero_edges = [e.index for e in g.es if e["sign"] == 0]
triangle_edges = []
for tri in triangles:
    n1, n2, n3 = tri
    e1 = g.get_eid(n1, n2)
    e2 = g.get_eid(n2, n3)
    e3 = g.get_eid(n3, n1)
    triangle_edges.append((e1, e2, e3))



In [149]:
def is_triangle_balanced(tri_edges, graph):
    prod = 1
    for e in tri_edges:
        s = graph.es[e]["sign"]
        if s == 0:
            return True  # unknown edges can still be assigned
        prod *= s
    return prod > 0


In [150]:
def backtrack_balance(graph, zero_edges, triangle_edges, idx=0):
    if idx == len(zero_edges):
        for tri in triangle_edges:
            if not is_triangle_balanced(tri, graph):
                return False
        return True

    edge_idx = zero_edges[idx]

    for sign in [1, -1]:        
        graph.es[edge_idx]["sign"] = sign        
        valid = True
        for tri in triangle_edges:
            if edge_idx in tri and not is_triangle_balanced(tri, graph):
                valid = False
                break
        if valid:
            if backtrack_balance(graph, zero_edges, triangle_edges, idx + 1):
                return True        
        graph.es[edge_idx]["sign"] = 0
    return False


In [151]:
success = backtrack_balance(g, zero_edges, triangle_edges)
if success:
    print("Graph successfully balanced!")
else:
    print("No assignment can fully balance the graph with given constraints.")


Graph successfully balanced!


# Part 2

In [323]:

def draw_clusters(graph, membership, graph_name,path):
    num_clusters = len(set(membership))
    palette = ig.drawing.colors.ClusterColoringPalette(num_clusters)
    graph.vs["color"] = [palette[m] for m in membership]
        
    graph.es["color"] = ["#3498db" if s > 0 else "#e74c3c" for s in graph.es["sign"]]
    graph.es["width"] = [2 if s > 0 else 1 for s in graph.es["sign"]]
    

    graph.vs["label"] = graph.vs["name"]
    graph.vs["label_size"] = 10
    
    layout = graph.layout("fr") 
    
    filename = os.path.join(path,f"clusters_{graph_name}.png")
    
    ig.plot(graph, filename, layout=layout, bbox=(600, 600), margin=50)
    print(f"Visualization saved as: {filename}")

def analyze_structural_balance(graph, graph_name, weak=False,path=""):
    print(f"\n--- Analyzing: {graph_name} ---")
    
    g_pos = graph.copy()
    neg_edge_indices = [e.index for e in graph.es if e['sign'] == -1]
    g_pos.delete_edges(neg_edge_indices)
    
    clusters = g_pos.components()
    membership = clusters.membership
    num_supernodes = len(clusters)
    print(f"Identified {num_supernodes} super-nodes (factions).")
        
    neg_edges = [e for e in graph.es if e['sign'] == -1]
    for edge in neg_edges:
        u, v = edge.tuple
        if membership[u] == membership[v]:
            u_name = graph.vs[u]['name']
            v_name = graph.vs[v]['name']
            print("Result: UNBALANCED")
            print(f"Reason: Internal contradiction. '{u_name}' and '{v_name}' are friends but have a negative edge.")
            print_supernode_assignments(graph, membership)
            draw_clusters(graph, membership, graph_name,path)
            return False

    reduced_graph = ig.Graph(num_supernodes)
    reduced_edges = []
    for edge in neg_edges:
        u, v = edge.tuple
        if membership[u] != membership[v]:
            reduced_edges.append((membership[u], membership[v]))
            
    reduced_graph.add_edges(reduced_edges)
    reduced_graph.simplify()
    
    is_bipartite = reduced_graph.is_bipartite()
    
    if is_bipartite:
        print("Result: Strongly BALANCED")
        print("Reason: Reduced graph is bipartite.")
    elif weak:
        print("Result: Weakly BALANCED")
        print("Reason: No inner-cluster negative edges (but reduced graph is not bipartite).")
    else:
        print("Result: UNBALANCED")
        print("Reason: Reduced graph contains an odd cycle.")

    print_supernode_assignments(graph, membership)
    draw_clusters(graph, membership, graph_name,path)
    return is_bipartite or weak

def print_supernode_assignments(graph, membership):
    print("\nSuper-node Assignments:")
    groups = defaultdict(list)
    for node_idx, cluster_id in enumerate(membership):        
        node_label = graph.vs[node_idx]['name'] 
        groups[cluster_id].append(node_label)
    
    for cluster_id, nodes in sorted(groups.items()):
        print(f"  Super-node {cluster_id} (Size {len(nodes)}): {nodes}")

In [ ]:
for i in ["a","b","c","d","e","f","g","h"]:
    df = pd.read_csv(f"data/Part_A/2/network_{i}.csv")
    graph = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
    graph.es["sign"] = df["sign"].tolist()
    analyze_structural_balance(graph,f"network_{i}",weak=False,path="./q4_data/q4_partb_images/")




--- Analyzing: network_a ---
Identified 3 super-nodes (factions).
Result: Strongly BALANCED
Reason: Reduced graph is bipartite.

Super-node Assignments:
  Super-node 0 (Size 28): [0, 2, 4, 6, 9, 27, 29, 1, 22, 26, 15, 31, 32, 33, 34, 3, 7, 8, 19, 24, 12, 20, 5, 16, 25, 11, 30, 13]
  Super-node 1 (Size 4): [10, 14, 21, 23]
  Super-node 2 (Size 3): [17, 18, 28]
Visualization saved as: ./q4_partb_images/clusters_network_a.png

--- Analyzing: network_b ---
Identified 3 super-nodes (factions).
Result: UNBALANCED
Reason: Internal contradiction. '0' and '14' are friends but have a negative edge.

Super-node Assignments:
  Super-node 0 (Size 36): [0, 3, 7, 9, 11, 14, 21, 28, 30, 34, 35, 36, 1, 13, 19, 32, 2, 5, 24, 20, 26, 27, 29, 31, 4, 37, 18, 10, 25, 33, 8, 12, 17, 23, 16, 22]
  Super-node 1 (Size 1): [6]
  Super-node 2 (Size 1): [15]
Visualization saved as: ./q4_partb_images/clusters_network_b.png

--- Analyzing: network_c ---
Identified 3 super-nodes (factions).
Result: UNBALANCED
Reason

# Part 3

In [327]:
for i in ["a","b","c","d","e"]:
    df = pd.read_csv(f"data/Part_A/3/network_{i}.csv")
    graph = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
    graph.es["sign"] = df["sign"].tolist()
    analyze_structural_balance(graph,f"network_{i}",weak=True,path="./q4_data/q4_partc_images/")




--- Analyzing: network_a ---
Identified 3 super-nodes (factions).
Result: Strongly BALANCED
Reason: Reduced graph is bipartite.

Super-node Assignments:
  Super-node 0 (Size 5): [4, 7, 12, 2, 0]
  Super-node 1 (Size 5): [1, 3, 13, 11, 10]
  Super-node 2 (Size 4): [9, 5, 8, 6]
Visualization saved as: ./q4_data/q4_partc_images/clusters_network_a.png

--- Analyzing: network_b ---
Identified 4 super-nodes (factions).
Result: Weakly BALANCED
Reason: No inner-cluster negative edges (but reduced graph is not bipartite).

Super-node Assignments:
  Super-node 0 (Size 5): [14, 10, 9, 0, 8]
  Super-node 1 (Size 5): [5, 15, 12, 18, 2]
  Super-node 2 (Size 5): [7, 6, 13, 17, 3]
  Super-node 3 (Size 5): [19, 1, 11, 16, 4]
Visualization saved as: ./q4_data/q4_partc_images/clusters_network_b.png

--- Analyzing: network_c ---
Identified 4 super-nodes (factions).
Result: Weakly BALANCED
Reason: No inner-cluster negative edges (but reduced graph is not bipartite).

Super-node Assignments:
  Super-node 0

# Part 4

In [329]:
df = pd.read_csv(f"data/Part_A/4/network_line_index.csv")
graph = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
graph.es["sign"] = df["sign"].tolist()

assert len(graph.es) == len(df["sign"]) 


In [404]:

def assign_random(graph: ig.Graph, number_of_clusters: int):
    clusters: list[list[int]] = [[] for i in range(number_of_clusters)]
    for v in graph.vs:
        choice = np.random.choice(number_of_clusters)
        clusters[choice].append(v.index)
        
    return clusters

In [405]:

def find_edge_between_clusters(graph:ig.Graph,clusterA:list[int],clusterB:list[int],sign=-1):
    count = 0
    for i in range(len(clusterA)):
        for j in range(len(clusterB)):
            v1,v2 = clusterA[i],clusterB[j]
            eid = graph.get_eid(v1,v2,error=False)
            if eid== -1 : continue
            else:
                edge = graph.es[eid]
                if edge["sign"]==sign : count+=1
    return count

def calculate_line_index(count_positive,count_negative,alpha):
    return alpha * count_positive + (1-alpha) * count_negative

In [482]:
def fix_clusters(graph: ig.Graph, clusters: list[list[int]], alpha=0.5):
    fixed = False
    while not fixed:
        fixed = True
        for i in range(len(clusters)):
            for nodei in list(clusters[i]): 
                best_index = i
                min_cost = float("inf")
                for j in range(len(clusters)):
                    pos_outside = 0
                    neg_inside = 0
                    for v_idx in graph.vs.indices:
                        if v_idx == nodei: continue
                        
                        eid = graph.get_eid(nodei, v_idx, error=False)
                        if eid == -1: continue
                        
                        edge_sign = graph.es[eid]["sign"]
                        is_in_cluster_j = v_idx in clusters[j]
                        
                        if is_in_cluster_j and edge_sign == -1:
                            neg_inside += 1
                        elif not is_in_cluster_j and edge_sign == 1:
                            pos_outside += 1
                    
                    current_cost = (alpha * pos_outside) + ((1 - alpha) * neg_inside)
                    
                    if current_cost < min_cost:
                        min_cost = current_cost
                        best_index = j

                if best_index != i:
                    fixed = False
                    clusters[i].remove(nodei)
                    clusters[best_index].append(nodei)

In [483]:

clusters = assign_random(graph, 4)
p_count = 0 
n_count = 0 
for i in range(len(clusters)):
    n_count += find_edge_between_clusters(graph, clusters[i], clusters[i], sign=-1) / 2
    for j in range(i + 1, len(clusters)):
        
        p_count += find_edge_between_clusters(graph, clusters[i], clusters[j], sign=1)

print("Initial Random Assignment")
print(f"P (Positive edges between clusters): {p_count}")
print(f"N (Negative edges within clusters):  {n_count}")
result_initial = calculate_line_index(p_count, n_count, alpha=0.5)
print(f"Initial Line Index: {result_initial}")
print("-" * 35)

fix_clusters(graph, clusters)
p_count = 0 
n_count = 0 
for i in range(len(clusters)):
    n_count += find_edge_between_clusters(graph, clusters[i], clusters[i], sign=-1) / 2
    for j in range(i + 1, len(clusters)):
        p_count += find_edge_between_clusters(graph, clusters[i], clusters[j], sign=1)

print("After Cluster Optimization (Local Search)")
print(f"P (Positive edges between clusters): {p_count}")
print(f"N (Negative edges within clusters):  {n_count}")
result_final = calculate_line_index(p_count, n_count, alpha=0.5)
print(f"Final Line Index: {result_final}")

improvement = result_initial - result_final
print(f"Total reduction in frustration: {improvement}")

Initial Random Assignment
P (Positive edges between clusters): 66
N (Negative edges within clusters):  13.0
Initial Line Index: 39.5
-----------------------------------
After Cluster Optimization (Local Search)
P (Positive edges between clusters): 0
N (Negative edges within clusters):  0.0
Final Line Index: 0.0
Total reduction in frustration: 39.5


# Part 5

In [3]:
df = pd.read_csv(f"data/Part_A/5/network_transitivity.csv")
graph = ig.Graph.TupleList(df.itertuples(index=False), directed=True)


In [45]:
def analyze_transitivity(graph:ig.Graph):
    transistive_triads = 0
    non_transistive_triads = 0
    missing_edges = 0
    for i in range(len(graph.vs)):
        nodei = graph.vs[i]
        for j in range(i+1,len(graph.vs)):
            nodej = graph.vs[j]
            for k in range(j+1,len(graph.vs)):
                nodek = graph.vs[k]
                transistive_flag = True
                nodes = [nodei,nodej,nodek]
                for nodea,nodeb,nodec in itertools.permutations(nodes):
                    if graph.are_adjacent(nodea,nodeb) and graph.are_adjacent(nodeb,nodec):
                        if not graph.are_adjacent(nodea,nodec):
                            transistive_flag = False
                            missing_edges+=1
                if transistive_flag:
                    transistive_triads+=1
                else :
                    non_transistive_triads+=1                    
    return transistive_triads,non_transistive_triads,missing_edges



def fix_transitivity(g:ig.Graph):
    added_edges = 0
    fixed = False
    graph:ig.Graph = g.copy()
    while not fixed:  
        fixed = True
        for i in range(len(graph.vs)):
            nodei = graph.vs[i]
            for j in range(i+1,len(graph.vs)):
                nodej = graph.vs[j]
                for k in range(j+1,len(graph.vs)):
                    nodek = graph.vs[k]
                    nodes = [nodei,nodej,nodek]
                    for nodea,nodeb,nodec in itertools.permutations(nodes):
                        if graph.are_adjacent(nodea,nodeb) and graph.are_adjacent(nodeb,nodec):
                            if not graph.are_adjacent(nodea,nodec):
                                fixed = False
                                graph.add_edge(nodea,nodec)
                                added_edges+=1

    return graph,added_edges




In [46]:
print(f"Original Graph has {len(graph.es)} Edges")
transistive_triads,non_transistive_triads,missing_edges = analyze_transitivity(graph)
print("Transistive Pairs: ",transistive_triads)
print("Non-Transistive Pairs: ",non_transistive_triads)
print("Missing Edges: ",missing_edges)
transistivity_ratio = (transistive_triads)/(transistive_triads + non_transistive_triads)
print("Transistivity Ratio: ", transistivity_ratio)
new_graph,added_edges = fix_transitivity(graph)

print(30*"-")
print(f"Transistivity Fixed with {added_edges} edges added")
transistive_triads,non_transistive_triads,missing_edges = analyze_transitivity(new_graph)
print("New Graph Transistive Pairs: ",transistive_triads)
print("New Graph Non-Transistive Pairs: ",non_transistive_triads)
print("New Graph Missing Edges: ",missing_edges)
transistivity_ratio = (transistive_triads)/(transistive_triads + non_transistive_triads)
print("New Graph Transistivity Ratio: ", transistivity_ratio)
print(f"New Graph has {len(new_graph.es)} Edges")


Original Graph has 820 Edges
Transistive Pairs:  1310223
Non-Transistive Pairs:  3177
Missing Edges:  3207
Transistivity Ratio:  0.997581087254454
------------------------------
Transistivity Fixed with 37991 edges added
New Graph Transistive Pairs:  1313400
New Graph Non-Transistive Pairs:  0
New Graph Missing Edges:  0
New Graph Transistivity Ratio:  1.0
New Graph has 38811 Edges


In [32]:
len(graph.vs)

200